<a href="https://colab.research.google.com/github/jemish13311331/CUAD-Fine-tunning/blob/main/CUAD_RoBERTa_Training_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUAD extractive-QA fine-tuning

This notebook fine-tunes `deepset/roberta-base-squad2` on the official CUAD data.

Workflow:

1. Download and parse CUAD.
2. Randomly select 20 contracts with seed 42.
3. Use 16 contracts for training and 4 different contracts for validation.
4. Tokenize long contracts with overlapping windows.
5. Fine-tune an extractive question-answering model.
6. report validation loss, Exact Match, and token F1.
7. Save and zip the best model.

The official CUAD test split is loaded but is **not used during model development**.

## 1. Install compatible packages

Run this cell once. If Colab asks you to restart the runtime, restart it and then continue with section 2. `peft` is not needed because this notebook performs ordinary full fine-tuning.

In [ ]:
%pip uninstall -y peft
%pip install -q -U 'transformers>=4.46,<5' 'datasets>=3,<6' 'accelerate>=1,<2' 'pyarrow>=19,<24'

## 2. Imports and GPU check

In [ ]:
import io
import json
import math
import random
import re
import string
import zipfile
from collections import Counter

import numpy as np
import requests
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    DefaultDataCollator,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

SEED = 42
set_seed(SEED)

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

if not torch.cuda.is_available():
    print("Warning: In Colab, select Runtime > Change runtime type > GPU.")

PyTorch: 2.14.1+cu130
GPU available: True
GPU: Tesla T4


## 3. Download and parse the official CUAD data

In [ ]:
CUAD_URL = "https://github.com/TheAtticusProject/cuad/raw/main/data.zip"

response = requests.get(CUAD_URL, timeout=120)
response.raise_for_status()
zip_file = zipfile.ZipFile(io.BytesIO(response.content))


def find_file(filename):
    # Find a filename even when it is nested inside the ZIP.
    matches = [path for path in zip_file.namelist() if path.endswith(filename)]
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} inside the CUAD ZIP.")
    return matches[0]


def load_cuad_split(filename):
    # Convert CUAD's SQuAD-style JSON into Hugging Face rows.
    raw_data = json.loads(zip_file.read(find_file(filename)))
    rows = []

    for document in raw_data["data"]:
        title = document.get("title", "").strip()

        for paragraph in document["paragraphs"]:
            # Do not strip context or answer text: answer_start uses exact character offsets.
            context = paragraph["context"]

            for qa in paragraph["qas"]:
                rows.append({
                    "id": qa["id"],
                    "title": title,
                    "context": context,
                    "question": qa["question"].strip(),
                    "answers": {
                        "text": [answer["text"] for answer in qa.get("answers", [])],
                        "answer_start": [
                            answer["answer_start"] for answer in qa.get("answers", [])
                        ],
                    },
                })

    return Dataset.from_list(rows)


dataset = DatasetDict({
    "train": load_cuad_split("train_separate_questions.json"),
    "test": load_cuad_split("test.json"),
})

print(dataset)
print("Columns:", dataset["train"].column_names)
print("Training contracts:", len(set(dataset["train"]["title"])))
print("Test contracts:", len(set(dataset["test"]["title"])))

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 22450
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 4182
    })
})
Columns: ['id', 'title', 'context', 'question', 'answers']
Training contracts: 408
Test contracts: 102


## 4. Explore the training data

In [ ]:
example = dataset["train"][0]

print("ID:", example["id"])
print("Title:", example["title"])
print("Question:", example["question"])
print("Answers:", example["answers"])
print("Context preview:", example["context"][:800], "...")

answerable = sum(bool(row["answers"]["text"]) for row in dataset["train"])
total = len(dataset["train"])
print(f"Answerable rows: {answerable:,} ({answerable / total:.1%})")
print(f"No-answer rows: {total - answerable:,} ({(total - answerable) / total:.1%})")

ID: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name_0
Title: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
Question: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
Answers: {'answer_start': [44], 'text': ['DISTRIBUTOR AGREEMENT']}
Context preview: EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may en

## 5. Select contracts and create train/validation sets

The split is performed by `title` (contract), not by row. This avoids data leakage when one contract contains many questions.

In [ ]:
NUM_CONTRACTS = 4       # Reduced from 20 to prevent running out of RAM
NUM_VALIDATION_CONTRACTS = 1 #reduced from 4 to prevent .........

all_titles = sorted(set(dataset["train"]["title"]))
rng = random.Random(SEED)
rng.shuffle(all_titles)

if NUM_CONTRACTS is None:
    selected_titles = all_titles
    num_validation = max(1, round(0.20 * len(selected_titles)))
else:
    selected_titles = all_titles[:min(NUM_CONTRACTS, len(all_titles))]
    num_validation = min(NUM_VALIDATION_CONTRACTS, len(selected_titles) - 1)

validation_titles = set(selected_titles[:num_validation])
training_titles = set(selected_titles[num_validation:])

train_raw = dataset["train"].filter(lambda row: row["title" ] in training_titles)
validation_raw = dataset["train"].filter(lambda row: row["title"] in validation_titles)
test_raw = dataset["test"]  # Keep untouched until final evaluation.

assert training_titles.isdisjoint(validation_titles)

print(f"Training:   {len(training_titles)} contracts, {len(train_raw):,} question rows")
print(f"Validation: {len(validation_titles)} contracts, {len(validation_raw):,} question rows")
print(f"Test:       {len(set(test_raw['title']))} contracts, {len(test_raw):,} question rows")
print("Validation contracts:")
for title in sorted(validation_titles):
    print(" -", title)

Filter:   0%|          | 0/22450 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22450 [00:00<?, ? examples/s]

Training:   3 contracts, 179 question rows
Validation: 1 contracts, 80 question rows
Test:       102 contracts, 4,182 question rows
Validation contracts:
 - FuelcellEnergyInc_20191106_8-K_EX-10.1_11868007_EX-10.1_Development Agreement


## 6. Load the tokenizer and model

In [ ]:
MODEL_NAME = "deepset/roberta-base-squad2"
MAX_LENGTH = 384
DOC_STRIDE = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

print("Loaded:", MODEL_NAME)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded: deepset/roberta-base-squad2


## 7. Tokenize and create start/end labels

Long contexts are split into overlapping windows. For each window, the label is either the answer's token span or the CLS token when the answer is not present in that window.

In [ ]:
def prepare_train_features(examples):
    questions = [question.lstrip() for question in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    start_positions = []
    end_positions = []

    for feature_index, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][feature_index]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(feature_index)
        sample_index = sample_mapping[feature_index]
        answers = examples["answers"][sample_index]

        # CUAD/SQuAD2 can contain questions with no answer.
        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # Use the first gold answer when multiple equivalent annotations exist.
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # If the answer is outside this window, label this window as no-answer.
        if offsets[token_start_index][0] > start_char or offsets[token_end_index][1] < end_char:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
            token_start_index += 1
        start_positions.append(token_start_index - 1)

        while offsets[token_end_index][1] >= end_char:
            token_end_index -= 1
        end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


train_features = train_raw.map(
    prepare_train_features,
    batched=True,
    remove_columns=train_raw.column_names,
    desc="Tokenizing training set",
)

validation_features = validation_raw.map(
    prepare_train_features,
    batched=True,
    remove_columns=validation_raw.column_names,
    desc="Tokenizing validation set",
)

print("Training windows:", len(train_features))
print("Validation windows:", len(validation_features))

Tokenizing training set:   0%|          | 0/179 [00:00<?, ? examples/s]

Tokenizing validation set:   0%|          | 0/80 [00:00<?, ? examples/s]

Training windows: 21178
Validation windows: 9127


## 8. Configure and run fine-tuning

In [ ]:
OUTPUT_DIR = "/content/cuad-roberta-qa"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_ratio=0.10,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_features,
    eval_dataset=validation_features,
    data_collator=DefaultDataCollator(),
    tokenizer=tokenizer,
)

train_result = trainer.train()
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

/tmp/ipykernel_1734/1447991999.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.047300,0.069797
2,0.012400,0.061533


***** train metrics *****
  epoch                    =        2.0
  total_flos               =  7730549GF
  train_loss               =     0.0462
  train_runtime            = 0:19:10.20
  train_samples_per_second =     36.825
  train_steps_per_second   =      4.604


## 9. Validation loss

In [ ]:
validation_result = trainer.evaluate()
trainer.log_metrics("validation", validation_result)
trainer.save_metrics("validation", validation_result)

print("Validation loss:", validation_result["eval_loss"])

***** validation metrics *****
  epoch                   =        2.0
  eval_loss               =     0.0615
  eval_runtime            = 0:00:55.82
  eval_samples_per_second =    163.502
  eval_steps_per_second   =      20.44
Validation loss: 0.06153294816613197


## 10. Exact Match and token F1 on validation questions

This evaluates final extracted answer text. Set `MAX_METRIC_EXAMPLES = None` to evaluate every validation row; using 100 is faster for an initial experiment.

In [ ]:
def normalize_answer(text):
    text = text.lower()
    text = "".join(character for character in text if character not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match_score(prediction, reference):
    return float(normalize_answer(prediction) == normalize_answer(reference))


def token_f1_score(prediction, reference):
    prediction_tokens = normalize_answer(prediction).split()
    reference_tokens = normalize_answer(reference).split()

    if not prediction_tokens and not reference_tokens:
        return 1.0
    if not prediction_tokens or not reference_tokens:
        return 0.0

    common = Counter(prediction_tokens) & Counter(reference_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0

    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


qa = pipeline(
    "question-answering",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

MAX_METRIC_EXAMPLES = 100  # Change to None for the full validation set.
metric_rows = validation_raw if MAX_METRIC_EXAMPLES is None else validation_raw.select(
    range(min(MAX_METRIC_EXAMPLES, len(validation_raw)))
)

exact_matches = []
f1_scores = []

for row in metric_rows:
    prediction = qa(
        question=row["question"],
        context=row["context"],
        max_seq_len=MAX_LENGTH,
        doc_stride=DOC_STRIDE,
        max_answer_len=128,
        handle_impossible_answer=True,
    )["answer"]

    references = row["answers"]["text"] or [""]
    exact_matches.append(max(exact_match_score(prediction, ref) for ref in references))
    f1_scores.append(max(token_f1_score(prediction, ref) for ref in references))

print(f"Evaluated validation questions: {len(metric_rows):,}")
print(f"Exact Match: {100 * np.mean(exact_matches):.2f}")
print(f"Token F1:    {100 * np.mean(f1_scores):.2f}")

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Evaluated validation questions: 80
Exact Match: 26.25
Token F1:    26.25


## 11. Save and download the best model

In [ ]:
FINAL_MODEL_DIR = "/content/cuad-roberta-qa-final"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Saved model to:", FINAL_MODEL_DIR)

Saved model to: /content/cuad-roberta-qa-final


In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    "/content/cuad-roberta-qa-final",
    "zip",
    FINAL_MODEL_DIR,
)
print("Created:", archive_path)
files.download(archive_path)

Created: /content/cuad-roberta-qa-final.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Try the trained model on a new contract

In [ ]:
new_context = (
    "The tenant may terminate this lease by providing the landlord with sixty days' "
    "written notice. An early termination fee of two months' rent will apply."
)

new_question = "What is the early termination fee?"

result = qa(
    question=new_question,
    context=new_context,
    max_seq_len=MAX_LENGTH,
    doc_stride=DOC_STRIDE,
    handle_impossible_answer=True,
)

print("Answer:", result["answer"])
print("Confidence:", round(result["score"], 4))
print("Character range:", result["start"], "to", result["end"])

Answer: 
Confidence: 0.9749
Character range: 0 to 0
